# 09 — Retry Policies, Rate Limiting, and Idempotency

Reliable clients handle transient failures without amplifying outages or duplicating actions. Practice bounded retries, exponential backoff, rate limits, and idempotency keys using simulated services.

*Advanced · 90–150 min*

## You will learn
- Differentiate transient from permanent failures
- Implement bounded retries with injected sleep
- Use rate-limiting and concurrency controls
- Explain idempotency keys and retry safety

## Examples

### 1. Classify failures before retrying

A retry is useful only when another attempt might succeed. Syntax errors, validation errors, and authentication errors generally require fixes, while select timeouts and server failures may be transient. Always cap attempts.

In [ ]:
class TransientError(Exception):
    pass
class PermanentError(Exception):
    pass

def unreliable():
    unreliable.calls += 1
    if unreliable.calls < 3:
        raise TransientError("temporary")
    return "ok"
unreliable.calls = 0
print("Calls needed to succeed:", 3)

**Think:** Why would retrying a bad password repeatedly be harmful?

### 2. Bounded exponential backoff

Backoff spaces out retries to avoid overwhelming a failing service. Inject a sleep function to test without waiting. In production, use jitter and obey server Retry-After semantics when applicable.

In [ ]:
def retry(call, *, attempts=4, delay=0.1, sleep=lambda _: None):
    if attempts < 1:
        raise ValueError("attempts must be >= 1")
    for index in range(attempts):
        try:
            return call()
        except TransientError:
            if index + 1 == attempts:
                raise
            sleep(delay * (2 ** index))

unreliable.calls = 0
delays = []
print(retry(unreliable, sleep=delays.append), delays)
assert delays == [0.1, 0.2]

**Think:** How can exponential backoff still synchronize clients without jitter?

### 3. Rate limiting versus concurrency limiting

A semaphore limits simultaneous operations; it does not by itself enforce a requests-per-second rate. A token bucket or server-provided rate limit can bound throughput over time. The demo shows a concurrency cap offline.

In [ ]:
import asyncio

async def bounded(values, limit):
    semaphore = asyncio.Semaphore(limit)
    active, peak = 0, 0
    async def process(value):
        nonlocal active, peak
        async with semaphore:
            active += 1
            peak = max(peak, active)
            await asyncio.sleep(0)
            active -= 1
            return value * 2
    results = await asyncio.gather(*(process(v) for v in values))
    return results, peak

results, peak = asyncio.run(bounded(range(8), limit=3))
print(results, peak)
assert peak <= 3

**Think:** Why doesn't limiting simultaneous requests guarantee fewer than N requests per second?

### 4. Idempotent requests and deduplication

A duplicate payment operation is dangerous. An idempotency key lets a server associate retries with one logical action. Real systems must persist keys atomically with outcomes, or race conditions can still cause duplication.

In [ ]:
class FakePayments:
    def __init__(self):
        self.results = {}
        self.sequence = 0
    def charge(self, key, amount):
        if key in self.results:
            return self.results[key]
        if amount <= 0:
            raise ValueError("positive amount required")
        self.sequence += 1
        result = {"payment_id": self.sequence, "amount": amount}
        self.results[key] = result
        return result

payments = FakePayments()
first = payments.charge("order-100", 50)
second = payments.charge("order-100", 50)
print(first, second)
assert first == second

**Think:** Why must a real service use atomic storage for idempotency records?

## Exercises

**Exercise 1.** Add injected-jitter support to the backoff policy and test maximum attempts.

In [ ]:
# Exercise 1
# TODO: Write code and tests here.

**Exercise 2.** Implement a bounded async worker pool and report peak concurrency.

In [ ]:
# Exercise 2
# TODO: Write code and tests here.

**Exercise 3.** Design a simple token bucket using an injectable clock.

In [ ]:
# Exercise 3
# TODO: Write code and tests here.

**Exercise 4.** Write unit tests demonstrating why repeating a non-idempotent charge is unsafe.

In [ ]:
# Exercise 4
# TODO: Write code and tests here.

## Common mistakes

- Never retry forever, and don't retry every error class.
- A semaphore is not a per-second rate limiter.
- Idempotency must be enforced atomically across concurrent clients.

**Next:** [10 — Profiling, Memory, and Honest Benchmarking](10_profiling_memory_benchmarks.ipynb)